modifying xyz files (nodes and linkers with X dummy atoms) regarding their bond length for the input for PORMAKE

IMPORT

In [68]:
from rdkit import Chem
from rdkit.Geometry import Point3D
import ase.io
from io import StringIO
from rdkit.Chem import rdDetermineBonds
import glob
import os

FUNCTIONS

In [69]:
def replace_He_with_H(atoms): # for Single Bond X
    x_indices = []
    
    xyz_file = StringIO()
    
    for i, atom in enumerate(atoms):
        if atom.symbol == 'He':
            atom.symbol = 'H'  # Replace 'He' with 'H'
            x_indices.append(i)  # Store the index of the replaced atom
    ase.io.write(xyz_file, atoms, format='xyz')
    xyz_block = xyz_file.getvalue()

    xyz_file.close()
    
    return xyz_block, x_indices

In [70]:
def replace_Se_with_O(atoms): # for Double Bond X
    x_indices = []
    
    xyz_file = StringIO()
    
    for i, atom in enumerate(atoms):
        if atom.symbol == 'Se':
            atom.symbol = 'O'  # Replace 'Se' with 'O'
            x_indices.append(i)  # Store the index of the replaced atom
    ase.io.write(xyz_file, atoms, format='xyz')
    xyz_block = xyz_file.getvalue()

    xyz_file.close()
    
    return xyz_block, x_indices

In [71]:
mode = "Se"
if mode == "Se":
    xyz_files = glob.glob("raw_Se/*.xyz")
elif mode == "He":
    xyz_files = glob.glob("raw_He/*.xyz")
else:
    raise ValueError("mode must be 'Se' or 'He'")
molecules = [ase.io.read(file) for file in xyz_files]
out_molecules = []
xyz_molecules = []

for mol in molecules:
    if mode == "Se":
        out = replace_Se_with_O(mol)
    elif mode == "He":
        out = replace_He_with_H(mol)
    print(out[1])
    out_molecules.append(out[-1])
    xyz_cof = Chem.MolFromXYZBlock(out[0])
    print(f"MolFromXYZBlock returned None? {xyz_cof is None}")
    if xyz_cof is None:
        continue
    rdDetermineBonds.DetermineBonds(xyz_cof)
    xyz_cof.Debug()
    xyz_molecules.append(xyz_cof)

# out_molecules

[9, 10, 11]
MolFromXYZBlock returned None? False
[30, 31]
MolFromXYZBlock returned None? False
[30, 31]
MolFromXYZBlock returned None? False


Atoms:
	0 8 O chg: 0  deg: 1 exp: 2 imp: 0 hyb: SP2
	1 6 C chg: 0  deg: 3 exp: 4 imp: 0 hyb: SP2
	2 6 C chg: 0  deg: 3 exp: 4 imp: 0 hyb: SP2
	3 8 O chg: 0  deg: 1 exp: 2 imp: 0 hyb: SP2
	4 6 C chg: 0  deg: 3 exp: 4 imp: 0 hyb: SP2
	5 6 C chg: 0  deg: 3 exp: 4 imp: 0 hyb: SP2
	6 8 O chg: 0  deg: 1 exp: 2 imp: 0 hyb: SP2
	7 6 C chg: 0  deg: 3 exp: 4 imp: 0 hyb: SP2
	8 6 C chg: 0  deg: 3 exp: 4 imp: 0 hyb: SP2
	9 8 O chg: 0  deg: 1 exp: 2 imp: 0 hyb: SP2
	10 8 O chg: 0  deg: 1 exp: 2 imp: 0 hyb: SP2
	11 8 O chg: 0  deg: 1 exp: 2 imp: 0 hyb: SP2
Bonds:
	0 1->0 order: 2 conj?: 1
	1 2->1 order: 1 conj?: 1
	2 4->3 order: 2 conj?: 1
	3 4->2 order: 1 conj?: 1
	4 5->4 order: 1 conj?: 1
	5 7->6 order: 2 conj?: 1
	6 7->5 order: 1 conj?: 1
	7 8->7 order: 1 conj?: 1
	8 8->1 order: 1 conj?: 1
	9 9->8 order: 2 conj?: 1
	10 10->5 order: 2 conj?: 1
	11 11->2 order: 2 conj?: 1
Atoms:
	0 7 N chg: 0  deg: 3 exp: 3 imp: 0 hyb: SP2
	1 6 C chg: 0  deg: 3 exp: 4 imp: 0 hyb: SP2
	2 6 C chg: 0  deg: 3 exp: 4 im

In [72]:
for xyz_cof, out in zip(xyz_molecules, out_molecules):
    for atom in xyz_cof.GetAtoms():
        if atom.GetIdx() in out:
            atom.SetIsotope(2)

In [73]:
filenames = [os.path.splitext(file)[0] + ".xyz" for file in xyz_files]
filenames[0]

# Define a subfolder for the output files
output_folder = "out_test"

# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

In [74]:
# Example: Iterate over each molecule and its corresponding original filename
for xyz_cof, xyz_filename in zip(xyz_molecules, filenames):
    # Get the base filename (without the path) to keep the same name
    base_filename = os.path.basename(xyz_filename)
    
    # Create the full path to the new corrected file in the new folder
    new_xyz_filename = os.path.join(output_folder, base_filename)
    
    with open(new_xyz_filename, "w") as xyz_file:
        num_atoms = xyz_cof.GetNumAtoms()
        xyz_file.write(f"{num_atoms}\n")
        xyz_file.write(f"Molecule\n")

        for atom in xyz_cof.GetAtoms():
            pos = xyz_cof.GetConformer().GetAtomPosition(atom.GetIdx())
            if atom.GetIsotope() == 2:  # Replace [Dummy] with X
                symbol = "X"
                # Get neighbors
                neighbors = [a.GetIdx() for a in atom.GetNeighbors()]
                if neighbors:
                    connected_atom_idx = neighbors[0]
                    connected_atom_pos = xyz_cof.GetConformer().GetAtomPosition(connected_atom_idx)
                    
                    # Calculate the vector from the connected atom to the X atom
                    vector = Point3D(pos.x - connected_atom_pos.x, 
                                     pos.y - connected_atom_pos.y, 
                                     pos.z - connected_atom_pos.z)
                    
                    # Define a scaling factor to adjust bond length (e.g., 1.2 for 20% longer)
                    scaling_factor = 0.8
                    
                    # Scale the vector
                    vector.x *= scaling_factor
                    vector.y *= scaling_factor
                    vector.z *= scaling_factor
                    
                    # Set the new position for the X atom
                    pos = Point3D(connected_atom_pos.x + vector.x,
                                  connected_atom_pos.y + vector.y,
                                  connected_atom_pos.z + vector.z)

            else:
                symbol = atom.GetSymbol()
            
            # Write atom information to the XYZ file
            xyz_file.write(f"{symbol} {pos.x} {pos.y} {pos.z}\n")

        # Write bonds in XYZ file
        for bond in xyz_cof.GetBonds():
            atom1, atom2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            bond_type = bond.GetBondTypeAsDouble()

            if bond_type == 1.0:
                bond_type_str = "S"
            elif bond_type == 1.5 and bond.IsInRing():
                bond_type_str = "A"
            elif bond_type == 1.5 and not bond.IsInRing():
                bond_type_str = "D"
            elif bond_type == 2.0:
                bond_type_str = "D"
            elif bond_type == 3.0:
                bond_type_str = "T"
            else:
                bond_type_str = "S"

            xyz_file.write(f"{atom1}    {atom2}    {bond_type_str}\n")

    print(f"XYZ file created: {new_xyz_filename}")

XYZ file created: out_test/Tp.xyz
XYZ file created: out_test/Azo.xyz
XYZ file created: out_test/DAAQ.xyz
